# Fine-tune WASB para detección de pelota — spain-france

Fine-tunea el detector temporal WASB (soccer) con los frames anotados con
`label_ball.py`. **Requiere GPU** (Entorno de ejecución → GPU).

⚠️ El training de WASB viene DESHABILITADO en el repo (el `Trainer` está
comentado). Este notebook lo re-habilita; puede necesitar ajustes menores.

## 1. Clonar WASB + habilitar training + pesos soccer

In [ ]:
%cd /content
!git clone -q https://github.com/nttcom/WASB-SBDT.git
%cd WASB-SBDT/src
!pip install -q omegaconf hydra-core gdown
# re-habilitar el Trainer (viene comentado)
import re, io
p="runners/__init__.py"; s=io.open(p).read()
s=s.replace("# from .train_and_test import Trainer","from .train_and_test import Trainer")
s=s.replace("#'train': Trainer,","'train': Trainer,")
io.open(p,"w").write(s)
# el import de VideosInferenceRunner (opcional) esta roto en el repo: hacerlo opcional
tt="runners/train_and_test.py"; t=io.open(tt).read()
t=t.replace("from .inference_videos import VideosInferenceRunner",
            "try:\n    from .inference_videos import VideosInferenceRunner\nexcept Exception:\n    VideosInferenceRunner=None")
# cargar pesos base ANTES de entrenar (fine-tune): tras construir el modelo
t=t.replace("self._model = self._model.to(self._device)",
            "import torch as _t\n        _w=cfg['runner'].get('resume')\n        if _w:\n            self._model.load_state_dict(_t.load(_w)['model_state_dict']); print('fine-tune desde',_w)\n        self._model = self._model.to(self._device)",1)
io.open(tt,"w").write(t)
# pesos base soccer (fine-tune parte de aca)
!mkdir -p ../pretrained_weights
!gdown -q 1pg0MpMtKZ6ziYEr4oyfKYPOO3hjLw94l -O ../pretrained_weights/wasb_soccer_best.pth.tar
print("WASB listo, Trainer habilitado")

## 2. Tu repo (make_wasb_dataset + labels) y el video

In [ ]:
%cd /content
!git clone -q --branch events-model https://github.com/pipachiesa/ncf_event_tracker.git repo
# subí el video del clip a /content/video.mp4 (o desde Drive)
from google.colab import drive; drive.mount('/content/drive')
VIDEO="/content/drive/MyDrive/football_analytics/videos/spain-france-test3min.mp4"  # ajustá
LABELS="/content/repo/events_model/dataset/ball_gt/spain-france_ball_labels.csv"
import os; assert os.path.exists(VIDEO), "subí el video del clip"

## 3. Generar el dataset en formato WASB

In [ ]:
%cd /content
!python repo/events_model/make_wasb_dataset.py \
    --video "$VIDEO" --labels "$LABELS" \
    --out /content/wasb_ft/soccer --clip sf3min --stride 2
# WASB divide train/test por 'video' id; con un solo clip usamos el mismo para
# validar el flujo (sobreajuste esperado, sirve para ver que aprende).

## 4. Config de training (WASB no lo trae; lo escribimos)

In [ ]:
cfg = """
defaults:
  - _self_
  - runner: eval
  - dataset: soccer
  - model: wasb
  - dataloader: default
  - detector: tracknetv2
  - transform: default
  - tracker: online
  - loss: hm_wbce
  - optimizer: adam_multistep
runner:
  name: train
  max_epochs: 30
  device: cuda
  gpus: [0]
  test: True
  inference_video: False
  find_fp1_epochs: []
  fp1_filename: null
  best_model_name: best_model.pth.tar
  resume: /content/WASB-SBDT/pretrained_weights/wasb_soccer_best.pth.tar
dataset:
  root_dir: /content/wasb_ft/soccer
  train: {videos: [match], num_clip_ratio: 1.0}
  test:  {videos: [match], num_clip_ratio: 1.0}
output_dir: /content/wasb_out
seed: 1234
"""
open("/content/WASB-SBDT/src/configs/finetune.yaml","w").write(cfg)
print("config escrito")

In [ ]:
%cd /content/WASB-SBDT/src
!python main.py --config-name=finetune

## 5. Evaluar sobre los 10 frames GT del pipeline

Correr el modelo fine-tuneado sobre `clip-test_gate_huecos.json` (los 10 frames
donde el pipeline perdía la pelota) y ver cuántos recupera vs el WASB base (0/10).
Reusar el script de eval del scratchpad (`pnl_test.py` estilo): cargar
`/content/wasb_out/.../best_model.pth.tar`, correr sobre las tripletas, medir.